## Setup: Install Required Dependencies

Install the necessary Python packages for interacting with Google's Gemini API via LangChain.

# LEAN Error Fixing with LangChain + Gemini

This notebook starts a LEAN server, collects diagnostics for `.lean` files, and sends files + errors to Gemini (via LangChain) to propose fixes.

In [21]:
# Install required packages
!pip install -q langchain-google-genai google-generativeai

## Import Required Libraries

Import all necessary modules for:
- File system operations (Path)
- JSON-RPC communication with Lean server
- Type hints
- LangChain integration with Gemini

In [22]:
import os
import json
import time
import threading
import queue
import subprocess
from pathlib import Path
from typing import Dict, List, Any, Optional
from urllib.parse import urljoin, urlunparse, quote
from langchain_google_genai import ChatGoogleGenerativeAI

## Configuration Settings

Set up key configuration parameters:
- **MODEL_NAME**: The Gemini model to use for fixing errors
- **API_KEY**: Retrieved from environment variable `GEMINI_API_KEY`
- **PROJECT_ROOT**: Path to the Lean package directory
- **LEAN_SOURCE_DIR**: Path to the Lean source files
- **LEAN_SERVER_CMD**: Command to start the Lean LSP server

In [24]:
# Configuration
MODEL_NAME = "gemini-2.5-pro"
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("GEMINI_API_KEY is not set in the environment")

PROJECT_ROOT = Path("extracted/CML_Lean")
LEAN_SOURCE_DIR = PROJECT_ROOT / "CML_Lean"

# Lean server command (stdio LSP)
LEAN_SERVER_CMD = ["lake", "env", "lean", "--server"]

## Preflight Check: Verify Lean Installation

Verify that `lake` (Lean's build tool) and the Lean compiler are properly installed and accessible by running version checks.

In [25]:
# Preflight: verify lake + lean toolchain
def run_cmd(cmd: List[str], cwd: Path) -> str:
    result = subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{result.stderr}")
    return result.stdout.strip()

print(run_cmd(["lake", "--version"], PROJECT_ROOT))
print(run_cmd(["lake", "env", "lean", "--version"], LEAN_SOURCE_DIR))

Lake version 5.0.0-src+db93fe1 (Lean version 4.27.0)
Lean (version 4.27.0, x86_64-w64-windows-gnu, commit db93fe1608548721853390a10cd40580fe7d22ae, Release)
Lean (version 4.27.0, x86_64-w64-windows-gnu, commit db93fe1608548721853390a10cd40580fe7d22ae, Release)


## LSP Client Implementation

Define the core infrastructure for communicating with the Lean Language Server:

### `LspClient` Class
A minimal LSP (Language Server Protocol) client that:
- Communicates with Lean server via JSON-RPC over stdio
- Manages request/response matching using message IDs
- Collects notifications (like `publishDiagnostics`) in a queue
- Captures stderr output for debugging
- Optionally writes malformed messages to disk for debugging

### Helper Functions
- `path_to_uri()`: Convert file paths to LSP URI format
- `_diagnostics_signature()`: Create a signature for comparing diagnostic sets
- `start_lean_server()`: Initialize and configure a Lean server instance
- `collect_diagnostics()`: Open files in the server, wait for diagnostics to "settle", and collect them

In [26]:
# Minimal LSP (JSON-RPC) client to receive Lean diagnostics
class LspClient:
    def __init__(self, cmd: List[str], cwd: Path, *, debug_io: bool = False):
        self.proc = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=False,
            bufsize=0
        )
        self._id = 0
        self._responses: Dict[int, Any] = {}
        self._notifications: queue.Queue = queue.Queue()
        self._stderr_lines: List[str] = []
        self._debug_io = debug_io
        self._io_errors: List[str] = []
        self._reader_thread = threading.Thread(target=self._read_loop, daemon=True)
        self._stderr_thread = threading.Thread(target=self._read_stderr, daemon=True)
        self._reader_thread.start()
        self._stderr_thread.start()

    def _read_loop(self):
        buf = b""
        while True:
            chunk = self.proc.stdout.read(4096)
            if not chunk:
                return
            buf += chunk
            while True:
                header_end = buf.find(b"\r\n\r\n")
                if header_end == -1:
                    break
                header_bytes = buf[:header_end]
                rest = buf[header_end + 4 :]
                header_text = header_bytes.decode("utf-8", errors="replace")
                content_length = None
                for line in header_text.split("\r\n"):
                    if line.lower().startswith("content-length:"):
                        try:
                            content_length = int(line.split(":", 1)[1].strip())
                        except Exception:
                            content_length = None
                        break
                if content_length is None:
                    buf = buf[1:]
                    continue
                if len(rest) < content_length:
                    break
                body = rest[:content_length]
                buf = rest[content_length:]
                try:
                    msg = json.loads(body.decode("utf-8"))
                except Exception as e:
                    err = f"JSON decode failed: {e}. Body len={len(body)}"
                    self._io_errors.append(err)
                    if self._debug_io:
                        try:
                            Path("lean_lsp_bad_message.bin").write_bytes(body)
                        except Exception:
                            pass
                    continue
                if "id" in msg:
                    self._responses[msg["id"]] = msg
                else:
                    self._notifications.put(msg)

    def _read_stderr(self):
        while True:
            line = self.proc.stderr.readline()
            if not line:
                break
            self._stderr_lines.append(line.decode("utf-8", errors="ignore").rstrip())
            if len(self._stderr_lines) > 200:
                self._stderr_lines = self._stderr_lines[-200:]

    def _send(self, payload: Dict[str, Any]):
        data = json.dumps(payload).encode("utf-8")
        content = b"Content-Length: " + str(len(data)).encode("ascii") + b"\r\n\r\n" + data
        self.proc.stdin.write(content)
        self.proc.stdin.flush()

    def request(self, method: str, params: Dict[str, Any], timeout: float = 30.0) -> Dict[str, Any]:
        self._id += 1
        req_id = self._id
        self._send({"jsonrpc": "2.0", "id": req_id, "method": method, "params": params})
        start = time.time()
        while time.time() - start < timeout:
            if req_id in self._responses:
                return self._responses.pop(req_id)
            if self.proc.poll() is not None:
                stderr_tail = "\n".join(self._stderr_lines[-20:])
                io_tail = "\n".join(self._io_errors[-5:])
                raise RuntimeError(
                    f"Lean server exited early (code {self.proc.returncode}).\n"
                    f"IO errors:\n{io_tail}\n\nStderr:\n{stderr_tail}"
                )
            time.sleep(0.05)
        stderr_tail = "\n".join(self._stderr_lines[-20:])
        io_tail = "\n".join(self._io_errors[-5:])
        raise TimeoutError(f"No response for {method}.\nIO errors:\n{io_tail}\n\nStderr:\n{stderr_tail}")

    def notify(self, method: str, params: Dict[str, Any]):
        self._send({"jsonrpc": "2.0", "method": method, "params": params})

    def next_notification(self, timeout: float = 5.0) -> Optional[Dict[str, Any]]:
        try:
            return self._notifications.get(timeout=timeout)
        except queue.Empty:
            return None

    def close(self):
        try:
            self.proc.terminate()
        except Exception:
            pass

def path_to_uri(path: Path) -> str:
    return "file:///" + quote(str(path.resolve()).replace("\\", "/"))

def _diagnostics_signature(diags: List[Dict[str, Any]]) -> str:
    parts = []
    for d in diags:
        rng = d.get("range", {})
        start = rng.get("start", {})
        msg = d.get("message", "")
        sev = d.get("severity", None)
        parts.append(f"{start.get('line')}:{start.get('character')}|{sev}|{msg}")
    return "\n".join(parts)

def start_lean_server(project_root: Path, *, debug_io: bool = False) -> LspClient:
    client = LspClient(LEAN_SERVER_CMD, project_root, debug_io=debug_io)
    root_uri = path_to_uri(project_root)
    client.request("initialize", {
        "processId": None,
        "rootUri": root_uri,
        "workspaceFolders": [
            {"uri": root_uri, "name": project_root.name}
        ],
        "capabilities": {
            "workspace": {"workspaceFolders": True}
        },
        "initializationOptions": {
            "editDelay": 10,
            "hasWidgets": False,
            "documentFormatting": False,
            "maxNumberOfProblems": 10000
        }
    }, timeout=60.0)
    client.notify("initialized", {})
    return client

def collect_diagnostics(
    client: LspClient,
    file_paths: List[Path],
    per_file_timeout: float = 90.0,
    settle_seconds: float = 3.0,
    min_wait_seconds: float = 2.0,
 ) -> Dict[str, Optional[List[Dict[str, Any]]]]:
    """Collect diagnostics by waiting for publishDiagnostics to arrive and settle.

    Notes:
    - Lean can publish diagnostics multiple times per file as it elaborates imports.
    - If a file never receives publishDiagnostics within timeout, result is None.
    """
    diagnostics: Dict[str, Optional[List[Dict[str, Any]]]] = {}
    for path in file_paths:
        uri = path_to_uri(path)
        text = path.read_text(encoding="utf-8")
        client.notify("textDocument/didOpen", {
            "textDocument": {
                "uri": uri,
                "languageId": "lean",
                "version": 1,
                "text": text
            }
        })
        # didSave tends to trigger the same work VS Code does
        client.notify("textDocument/didSave", {
            "textDocument": {"uri": uri},
            "text": text
        })
        last_diags: Optional[List[Dict[str, Any]]] = None
        last_sig: Optional[str] = None
        last_change_time: Optional[float] = None
        start = time.time()
        while time.time() - start < per_file_timeout:
            msg = client.next_notification(timeout=1.0)
            now = time.time()
            if msg is None:
                # Ensure we wait at least a bit, even if idle
                if (now - start) < min_wait_seconds:
                    continue
            else:
                method = msg.get("method")
                if method == "textDocument/publishDiagnostics":
                    params = msg.get("params", {})
                    if params.get("uri") == uri:
                        current = params.get("diagnostics", [])
                        sig = _diagnostics_signature(current)
                        if sig != last_sig:
                            last_sig = sig
                            last_diags = current
                            last_change_time = now
                # Some servers send background progress; we just ignore it here but it keeps loop alive
            if last_change_time is not None and (now - last_change_time) >= settle_seconds and (now - start) >= min_wait_seconds:
                break
        diagnostics[str(path)] = last_diags
    return diagnostics

## Initial Diagnostic Collection

Collect diagnostics for all `.lean` files in the source directory:
1. Find all `.lean` files
2. Start a Lean server
3. Open each file and collect diagnostics
4. Display a summary of diagnostic counts per file

This provides the initial error state before any fixes are attempted.

In [27]:
# Collect diagnostics from all .lean files (debug_io=True writes bad frames to lean_lsp_bad_message.bin)
lean_files = sorted(LEAN_SOURCE_DIR.glob("*.lean"))
if not lean_files:
    raise FileNotFoundError(f"No .lean files found in {LEAN_SOURCE_DIR}")

client = start_lean_server(PROJECT_ROOT, debug_io=True)
diagnostics = collect_diagnostics(client, lean_files, per_file_timeout=90.0, settle_seconds=3.0)
client.close()

# Show a quick summary
for path, diags in diagnostics.items():
    if diags is None:
        print(f"{path}: no diagnostics received")
    else:
        print(f"{path}: {len(diags)} diagnostic(s)")

extracted\CML_Lean\CML_Lean\ast.lean: 1 diagnostic(s)
extracted\CML_Lean\CML_Lean\location.lean: 14 diagnostic(s)
extracted\CML_Lean\CML_Lean\namespace.lean: 31 diagnostic(s)
extracted\CML_Lean\CML_Lean\namespaceProps.lean: 1 diagnostic(s)


## Display Detailed Diagnostics for location.lean

Print detailed information about each diagnostic in `location.lean`:
- Severity level (ERROR, WARNING, INFO, HINT)
- Line and character position
- Error message

This helps understand what issues need to be fixed.

In [28]:
# Print detailed diagnostics for location.lean
location_lean_path = str(LEAN_SOURCE_DIR / "location.lean")

if location_lean_path in diagnostics:
    diags = diagnostics[location_lean_path]
    if diags is None:
        print(f"No diagnostics received for location.lean")
    elif len(diags) == 0:
        print(f"✓ location.lean has no errors!")
    else:
        print(f"Diagnostics for location.lean ({len(diags)} issue(s)):")
        print("=" * 80)
        for i, diag in enumerate(diags, 1):
            severity = diag.get("severity", "")
            severity_text = {1: "ERROR", 2: "WARNING", 3: "INFO", 4: "HINT"}.get(severity, f"SEVERITY_{severity}")
            
            rng = diag.get("range", {})
            start = rng.get("start", {})
            end = rng.get("end", {})
            line = start.get("line", 0) + 1  # LSP is 0-indexed, display as 1-indexed
            char = start.get("character", 0)
            
            message = diag.get("message", "")
            
            print(f"\n[{i}] {severity_text} at line {line}:{char}")
            print(f"    {message}")
else:
    print(f"location.lean not found in diagnostics")
    print(f"Available files: {list(diagnostics.keys())}")

Diagnostics for location.lean (14 issue(s)):

[1] WARNING at line 101:8
    declaration uses 'sorry'

[2] WARNING at line 107:8
    declaration uses 'sorry'

[3] WARNING at line 113:8
    declaration uses 'sorry'

[4] WARNING at line 119:8
    declaration uses 'sorry'

[5] WARNING at line 125:8
    declaration uses 'sorry'

[6] WARNING at line 131:8
    declaration uses 'sorry'

[7] WARNING at line 143:8
    declaration uses 'sorry'

[8] WARNING at line 149:8
    declaration uses 'sorry'

[9] WARNING at line 155:8
    declaration uses 'sorry'

[10] WARNING at line 191:8
    declaration uses 'sorry'

[11] WARNING at line 200:8
    declaration uses 'sorry'

[12] WARNING at line 208:8
    declaration uses 'sorry'

[13] WARNING at line 215:8
    declaration uses 'sorry'

[14] WARNING at line 221:8
    declaration uses 'sorry'


## AI Fixing Functions

### `fix_lean_file_with_gemini_iterative()`
**Main iterative fixing function** that:
- Runs up to `max_iterations` (default 5) fix-check cycles
- For each iteration:
  1. Formats diagnostics into a clear prompt
  2. Sends file + errors to Gemini for fixing
  3. Extracts and cleans the fixed code
  4. Writes fixed code to disk
  5. Re-runs Lean diagnostics to check for remaining errors
  6. Stops if no errors remain (SUCCESS)
  7. Otherwise continues to next iteration
- Returns: `(final_code, iterations_used, success_flag)`
- Tracks progress and warns if error count isn't decreasing

### `fix_lean_file_with_gemini()`
**Single-shot fixing function** (for backward compatibility):
- Sends file + diagnostics to Gemini once
- Returns fixed code without re-checking
- Useful for manual iteration or testing

In [29]:
def fix_lean_file_with_gemini_iterative(
    file_path: Path, 
    initial_diags: List[Dict[str, Any]], 
    llm: ChatGoogleGenerativeAI,
    max_iterations: int = 5,
    per_file_timeout: float = 90.0,
    settle_seconds: float = 3.0
) -> tuple[str, int, bool]:
    """
    Iteratively fix a Lean file until no errors remain or max iterations reached.
    
    Args:
        file_path: Path to the .lean file
        initial_diags: Initial list of diagnostic dictionaries from LSP
        llm: LangChain ChatGoogleGenerativeAI instance
        max_iterations: Maximum number of fix-check cycles (default: 5)
        per_file_timeout: Timeout for diagnostic collection per file
        settle_seconds: Seconds to wait for diagnostics to settle
        
    Returns:
        tuple of (final_code, iterations_used, success)
    """
    current_diags = initial_diags
    
    for iteration in range(1, max_iterations + 1):
        print(f"\n{'='*80}")
        print(f"Iteration {iteration}/{max_iterations}: {len(current_diags)} diagnostic(s) to fix")
        print(f"{'='*80}")
        
        # Read current file content
        file_content = file_path.read_text(encoding="utf-8")
        
        # Format diagnostics for the prompt
        diagnostic_text = []
        for i, diag in enumerate(current_diags, 1):
            severity = diag.get("severity", "")
            severity_text = {1: "ERROR", 2: "WARNING", 3: "INFO", 4: "HINT"}.get(
                severity, f"SEVERITY_{severity}"
            )
            
            rng = diag.get("range", {})
            start = rng.get("start", {})
            line = start.get("line", 0) + 1  # Convert to 1-indexed
            char = start.get("character", 0)
            
            message = diag.get("message", "")
            
            diagnostic_text.append(f"[{i}] {severity_text} at line {line}:{char}\n    {message}")
        
        diagnostics_str = "\n\n".join(diagnostic_text)
        
        # Create the prompt
        iteration_note = f"\n\nNote: This is iteration {iteration} of fixing. " if iteration > 1 else ""
        prompt = f"""You are an expert in Lean 4 theorem proving. I have a Lean 4 file with compilation errors. Please fix ALL the issues and return the corrected file.{iteration_note}

**File: {file_path.name}**

**Current Content:**
```lean
{file_content}
```

**Diagnostics (Errors/Warnings):**
{diagnostics_str}

**Instructions:**
1. Fix ALL errors and warnings listed above
2. Preserve the original structure and logic as much as possible
3. Use correct Lean 4 syntax
4. Return ONLY the fixed Lean code, without any explanations or markdown code blocks
5. Ensure all imports, namespaces, and declarations are valid
6. If a type or function is undefined, check if it should be imported or defined
7. The comments in the code is the HOL4 counterpart for the Lean code, refer to them for logical correctness

Please provide the complete fixed file content:"""

        # Send to Gemini
        print(f"Sending to Gemini for fixing...")
        response = llm.invoke(prompt)
        
        # Extract the fixed code from response
        fixed_code = response.content.strip()
        
        # Remove markdown code blocks if present
        if fixed_code.startswith("```"):
            lines = fixed_code.split("\n")
            lines = lines[1:]  # Remove first line (```lean or ```)
            if lines and lines[-1].strip() == "```":
                lines = lines[:-1]  # Remove last line if it's ```
            fixed_code = "\n".join(lines)
        
        print(f"✓ Received fixed code ({len(fixed_code)} characters)")
        
        # Write the fixed code to file
        file_path.write_text(fixed_code, encoding="utf-8")
        print(f"✓ Written to {file_path.name}")
        
        # Re-run diagnostics to check if issues are resolved
        print(f"Re-checking diagnostics...")
        client_check = start_lean_server(PROJECT_ROOT, debug_io=False)
        diagnostics_result = collect_diagnostics(
            client_check,
            [file_path],
            per_file_timeout=per_file_timeout,
            settle_seconds=settle_seconds
        )
        client_check.close()
        
        # Get the new diagnostics
        new_diags = diagnostics_result.get(str(file_path))
        
        if new_diags is None:
            print(f"⚠ Warning: No diagnostics received (timeout or server issue)")
            return fixed_code, iteration, False
        
        if len(new_diags) == 0:
            print(f"\n{'✓'*40}")
            print(f"✓✓✓ SUCCESS! File is now error-free after {iteration} iteration(s)! ✓✓✓")
            print(f"{'✓'*40}")
            return fixed_code, iteration, True
        
        # Still have errors, prepare for next iteration
        print(f"Still has {len(new_diags)} diagnostic(s) remaining")
        
        # Check if we're making progress
        if len(new_diags) >= len(current_diags):
            print(f"⚠ Warning: Error count did not decrease (was {len(current_diags)}, now {len(new_diags)})")
        
        current_diags = new_diags
    
    # Max iterations reached
    print(f"\n{'='*80}")
    print(f"⚠ Max iterations ({max_iterations}) reached. {len(current_diags)} diagnostic(s) still remain.")
    print(f"{'='*80}")
    return fixed_code, max_iterations, False


def fix_lean_file_with_gemini(file_path: Path, diags: List[Dict[str, Any]], llm: ChatGoogleGenerativeAI) -> str:
    """
    Send a Lean file and its diagnostics to Gemini for fixing (single iteration).
    
    Args:
        file_path: Path to the .lean file
        diags: List of diagnostic dictionaries from LSP
        llm: LangChain ChatGoogleGenerativeAI instance
        
    Returns:
        Fixed Lean code as a string
    """
    # Read the current file content
    file_content = file_path.read_text(encoding="utf-8")
    
    # Format diagnostics for the prompt
    diagnostic_text = []
    for i, diag in enumerate(diags, 1):
        severity = diag.get("severity", "")
        severity_text = {1: "ERROR", 2: "WARNING", 3: "INFO", 4: "HINT"}.get(severity, f"SEVERITY_{severity}")
        
        rng = diag.get("range", {})
        start = rng.get("start", {})
        line = start.get("line", 0) + 1  # Convert to 1-indexed
        char = start.get("character", 0)
        
        message = diag.get("message", "")
        
        diagnostic_text.append(f"[{i}] {severity_text} at line {line}:{char}\n    {message}")
    
    diagnostics_str = "\n\n".join(diagnostic_text)
    
    # Create the prompt
    prompt = f"""You are an expert in Lean 4 theorem proving. I have a Lean 4 file with compilation errors. Please fix ALL the issues and return the corrected file.

**File: {file_path.name}**

**Current Content:**
```lean
{file_content}
```

**Diagnostics (Errors/Warnings):**
{diagnostics_str}

**Instructions:**
1. Fix ALL errors and warnings listed above
2. Preserve the original structure and logic as much as possible
3. Use correct Lean 4 syntax
4. Return ONLY the fixed Lean code, without any explanations or markdown code blocks
5. Ensure all imports, namespaces, and declarations are valid
6. If a type or function is undefined, check if it should be imported or defined
7. The comments in the code is the HOL4 counterpart for the Lean code, refer to them for logical correctness

Please provide the complete fixed file content:"""

    # Send to Gemini
    response = llm.invoke(prompt)
    
    # Extract the fixed code from response
    fixed_code = response.content.strip()
    
    # Remove markdown code blocks if present
    if fixed_code.startswith("```"):
        lines = fixed_code.split("\n")
        # Remove first line (```lean or ```)
        lines = lines[1:]
        # Remove last line if it's ```
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        fixed_code = "\n".join(lines)
    
    return fixed_code

## Initialize Gemini LLM

Create a LangChain `ChatGoogleGenerativeAI` instance:
- Model: `gemini-2.5-flash`
- Temperature: 0.1 (low for more deterministic/consistent fixes)
- API key from environment variable

In [30]:
# Initialize LangChain ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    google_api_key=API_KEY,
    temperature=0.1  # Lower temperature for more deterministic fixes
)

print(f"Initialized {MODEL_NAME} via LangChain")

Initialized gemini-2.5-pro via LangChain


## Test: Iterative Fixing on location.lean

**Main test cell** that demonstrates the full iterative fixing workflow:
1. Check if `location.lean` has diagnostics
2. Create a backup of the original file (if not exists)
3. Call `fix_lean_file_with_gemini_iterative()` with up to 5 iterations
4. Display progress for each iteration:
   - Diagnostic count
   - Gemini response status
   - Re-check results
5. Show final summary:
   - Number of iterations used
   - Success status (all errors fixed or max iterations reached)
   - Final code length

This is the recommended way to fix Lean files with automatic verification.

In [31]:
# Test iterative fixing on location.lean
location_file = LEAN_SOURCE_DIR / "location.lean"
location_diagnostics = diagnostics.get(str(location_file))

if location_diagnostics and len(location_diagnostics) > 0:
    print(f"Starting iterative fixing for location.lean")
    print(f"Initial diagnostics: {len(location_diagnostics)}")
    print("=" * 80)
    
    # Create backup if it doesn't exist
    backup_path = location_file.with_suffix(".lean.bak")
    if not backup_path.exists():
        import shutil
        shutil.copy2(location_file, backup_path)
        print(f"✓ Backed up original to: {backup_path.name}\n")
    
    try:
        final_code, iterations, success = fix_lean_file_with_gemini_iterative(
            location_file,
            location_diagnostics,
            llm,
            max_iterations=5,
            per_file_timeout=90.0,
            settle_seconds=3.0
        )
        
        print(f"\n{'='*80}")
        print(f"FINAL RESULT:")
        print(f"  Iterations used: {iterations}")
        print(f"  Success: {'YES ✓✓✓' if success else 'NO (max iterations reached)'}")
        print(f"  Final code length: {len(final_code)} characters")
        print(f"{'='*80}")
        
    except Exception as e:
        print(f"\n✗ Error during iterative fixing: {e}")
        import traceback
        traceback.print_exc()
else:
    print("No diagnostics to fix for location.lean (file may be error-free or not found)")

Starting iterative fixing for location.lean
Initial diagnostics: 14

Iteration 1/5: 14 diagnostic(s) to fix
Sending to Gemini for fixing...
✓ Received fixed code (11374 characters)
✓ Written to location.lean
Re-checking diagnostics...
✓ Received fixed code (11374 characters)
✓ Written to location.lean
Re-checking diagnostics...
Still has 20 diagnostic(s) remaining
⚠ Warning: Error count did not decrease (was 14, now 20)

Iteration 2/5: 20 diagnostic(s) to fix
Sending to Gemini for fixing...
Still has 20 diagnostic(s) remaining
⚠ Warning: Error count did not decrease (was 14, now 20)

Iteration 2/5: 20 diagnostic(s) to fix
Sending to Gemini for fixing...
✓ Received fixed code (12269 characters)
✓ Written to location.lean
Re-checking diagnostics...
Still has 8 diagnostic(s) remaining

Iteration 3/5: 8 diagnostic(s) to fix
Sending to Gemini for fixing...
✓ Received fixed code (12365 characters)
✓ Written to location.lean
Re-checking diagnostics...
Still has 27 diagnostic(s) remaining
⚠ Wa

KeyboardInterrupt: 

## Alternative: Single-Shot Fix (Manual Iteration)

**Alternative approach** using the single-shot fixing function:
1. Get diagnostics for `location.lean`
2. Send to Gemini for fixing (one time)
3. Display preview of fixed code (first 500 characters)
4. Create backup of original file
5. Write fixed code to disk

This approach requires manual verification and re-running if errors remain. Use this for:
- Testing individual fixes
- More control over the fixing process
- Inspecting intermediate results

In [14]:
# Test on location.lean
location_file = LEAN_SOURCE_DIR / "location.lean"
location_diagnostics = diagnostics.get(str(location_file))

if location_diagnostics and len(location_diagnostics) > 0:
    print(f"Sending location.lean with {len(location_diagnostics)} diagnostic(s) to Gemini...")
    print("=" * 80)
    
    try:
        fixed_code = fix_lean_file_with_gemini(location_file, location_diagnostics, llm)
        
        print("\n✓ Received fixed code from Gemini")
        print(f"Length: {len(fixed_code)} characters")
        print("\nFirst 500 characters of fixed code:")
        print("-" * 80)
        print(fixed_code[:500])
        print("-" * 80)
        
        # Optionally save the fixed code
        backup_path = location_file.with_suffix(".lean.bak")
        if not backup_path.exists():
            location_file.rename(backup_path)
            print(f"\n✓ Backed up original to: {backup_path.name}")
        
        location_file.write_text(fixed_code, encoding="utf-8")
        print(f"✓ Written fixed code to: {location_file.name}")
        
    except Exception as e:
        print(f"\n✗ Error during fixing: {e}")
        import traceback
        traceback.print_exc()
else:
    print("No diagnostics to fix for location.lean (file may be error-free or not found)")

Sending location.lean with 24 diagnostic(s) to Gemini...

✓ Received fixed code from Gemini
Length: 5900 characters

First 500 characters of fixed code:
--------------------------------------------------------------------------------
-- Auto-generated LEAN 4 file from HOL4 translation
-- Theory: location
-- Generated using Gemini API

namespace CML_Lean.location

/-
Original HOL4 Datatype: locn
locn = UNKNOWNpt | EOFpt | POSN num num
-/
inductive locn where
  | UNKNOWNpt : locn
  | EOFpt : locn
  | POSN : Nat → Nat → locn

/-
Original HOL4 Definition: locnrow_def
locnrow (POSN r c) = r
-/
def locnrow : locn → Nat
  | locn.POSN r _ => r
  | _ => 0

/-
Original HOL4 Definition: locn_rowupdate_def
locn_rowupdate f (POSN r c) = 
--------------------------------------------------------------------------------

✓ Backed up original to: location.lean.bak
✓ Written fixed code to: location.lean

✓ Received fixed code from Gemini
Length: 5900 characters

First 500 characters of fixed code:
-----

## Verification: Re-check Diagnostics After Fixing

**Manual verification cell** that:
1. Starts a fresh Lean server
2. Re-collects diagnostics for `location.lean`
3. Displays results:
   - ✓✓✓ SUCCESS if no errors remain
   - ⚠ Warning with detailed error list if issues persist

Use this cell after the single-shot fix to verify results, or after iterative fixing to double-check the final state.

In [15]:
# Re-run diagnostics on fixed location.lean to verify the fix
print("Re-running diagnostics on fixed location.lean...")
print("=" * 80)

# Start a new Lean server
client_verify = start_lean_server(PROJECT_ROOT, debug_io=True)

# Collect diagnostics for just location.lean
location_file_path = LEAN_SOURCE_DIR / "location.lean"
diagnostics_verify = collect_diagnostics(
    client_verify, 
    [location_file_path], 
    per_file_timeout=90.0, 
    settle_seconds=3.0
)

client_verify.close()

# Check results
location_diags_after = diagnostics_verify.get(str(location_file_path))

if location_diags_after is None:
    print("\n⚠ No diagnostics received (timeout or server issue)")
elif len(location_diags_after) == 0:
    print("\n✓✓✓ SUCCESS! location.lean now has NO errors or warnings! ✓✓✓")
else:
    print(f"\n⚠ Still has {len(location_diags_after)} diagnostic(s):")
    print("-" * 80)
    for i, diag in enumerate(location_diags_after, 1):
        severity = diag.get("severity", "")
        severity_text = {1: "ERROR", 2: "WARNING", 3: "INFO", 4: "HINT"}.get(severity, f"SEVERITY_{severity}")
        
        rng = diag.get("range", {})
        start = rng.get("start", {})
        line = start.get("line", 0) + 1
        char = start.get("character", 0)
        
        message = diag.get("message", "")
        
        print(f"\n[{i}] {severity_text} at line {line}:{char}")
        print(f"    {message}")

Re-running diagnostics on fixed location.lean...

⚠ Still has 10 diagnostic(s):
--------------------------------------------------------------------------------

[1] ERROR at line 97:2
    No goals to be solved

[2] WARNING at line 104:8
    declaration uses 'sorry'

[3] WARNING at line 111:8
    declaration uses 'sorry'

[4] WARNING at line 118:8
    declaration uses 'sorry'

[5] WARNING at line 125:8
    declaration uses 'sorry'

[6] WARNING at line 132:8
    declaration uses 'sorry'

[7] WARNING at line 184:10
    unused variable `h2`

Note: This linter can be disabled with `set_option linter.unusedVariables false`

[8] WARNING at line 202:8
    declaration uses 'sorry'

[9] WARNING at line 212:8
    declaration uses 'sorry'

[10] WARNING at line 221:8
    declaration uses 'sorry'

⚠ Still has 10 diagnostic(s):
--------------------------------------------------------------------------------

[1] ERROR at line 97:2
    No goals to be solved

[2] WARNING at line 104:8
    declaration 